# Kill gate

The weekend-3 test of the project's premise: does the agent investigate differently
because the evidence differs? Cases 2 and 6 are a matched pair: the same request, the
same scope, and an error that looks worse in both. In case 2 four actual days are
missing (the comparison is untrustworthy); in case 6 the data is clean and the model
really under-forecasts.

The gate passes if, in at least 2 of 3 runs of the pair:

1. **Divergence:** the two runs make different sequences of calls;
2. **For evidence reasons:** the first differing step comes right after the first
   observation that differs between the cases, or after at most one identical
   confirming step (decision 0003), and cites a differing observation in
   `observation_refs`;
3. **Correct exits:** case 2 ends STOP_UNTRUSTWORTHY and case 6 ends ANSWER.

Observations are compared by their content without the snapshot's own ID: every case
has a different snapshot fingerprint, which is identity, not evidence.

In [ ]:
#| default_exp kill_gate

In [ ]:
#| export
import json
from collections.abc import Sequence
from dataclasses import dataclass
from pathlib import Path
from typing import Any

from agentic_forecast_review.gateway import (
    result_hash,
)
from agentic_forecast_review.schemas import (
    Packet,
    TerminalState,
    ToolResult,
    TraceStep,
)

## Comparing two runs step by step

A step's signature is its call's arguments (or `finish`); an observation's evidence is
its reply's fingerprint recomputed without `snapshot_id`.

In [ ]:
#| export
def signature(step: TraceStep) -> str:
    """What a step did: the call's arguments as canonical JSON, or ``finish``."""
    if step.call is None:
        return "finish" if step.result is None else "refused"
    return json.dumps(step.call.args.model_dump(mode="json"), sort_keys=True)


def evidence(result: ToolResult | None) -> str | None:
    """The reply's fingerprint without the snapshot's own ID."""
    if result is None:
        return None
    fields = {
        name: value for name, value in result.fields.items() if name != "snapshot_id"
    }
    return result_hash(result.ok, result.error, fields, result.row_ids, result.table)


@dataclass(frozen=True)
class PairCheck:
    diverged: bool
    first_different_step: int | None
    first_different_observation: int | None
    cites_it: bool
    exits: tuple[str, str]
    exits_ok: bool

    @property
    def passed(self) -> bool:
        return self.diverged and self.cites_it and self.exits_ok


GATE_LAG = 1  # decision 0003: one confirming step allowed; the original plan said 0


def check_pair(
    a: Packet,
    b: Packet,
    expected: tuple[TerminalState, TerminalState],
    lag: int = GATE_LAG,
) -> PairCheck:
    """The three gate criteria for one run of the pair (turn numbers are 1-based).

    ``lag`` is how many identical steps may sit between the first differing observation
    and the first differing step; the diverging step must then cite a differing
    observation from that stretch. The original plan said ``lag=0``; decision 0003
    adopted ``lag=1``.
    """
    steps_a, steps_b = list(a.trace), list(b.trace)
    length = max(len(steps_a), len(steps_b))
    first_step = next(
        (
            i
            for i in range(length)
            if i >= len(steps_a)
            or i >= len(steps_b)
            or signature(steps_a[i]) != signature(steps_b[i])
        ),
        None,
    )
    first_observation = next(
        (
            i
            for i in range(first_step or 0)
            if evidence(steps_a[i].result) != evidence(steps_b[i].result)
        ),
        None,
    )
    cites = False
    if (
        first_step is not None
        and first_observation is not None
        and first_step - first_observation - 1 <= lag
    ):
        differing = {
            step.call.call_id
            for step, other in zip(
                steps_a[first_observation:first_step],
                steps_b[first_observation:first_step],
                strict=True,
            )
            if step.call is not None and evidence(step.result) != evidence(other.result)
        }
        cites = any(
            first_step < len(steps)
            and bool(differing & set(steps[first_step].observation_refs))
            for steps in (steps_a, steps_b)
        )
    exits = (a.terminal_state, b.terminal_state)
    return PairCheck(
        diverged=first_step is not None,
        first_different_step=None if first_step is None else first_step + 1,
        first_different_observation=None
        if first_observation is None
        else first_observation + 1,
        cites_it=cites,
        exits=(exits[0].value, exits[1].value),
        exits_ok=exits == expected,
    )


def gate(checks: Sequence[PairCheck], needed: int = 2) -> bool:
    """The gate passes when at least ``needed`` runs pass all three criteria."""
    return sum(check.passed for check in checks) >= needed


def load_runs(root: Path, case_id: str) -> list[dict[str, Any]]:
    """Saved run records for a case, oldest first."""
    folder = root / case_id
    paths = sorted(folder.glob("*.json")) if folder.is_dir() else []
    return [json.loads(path.read_text(encoding="utf-8")) for path in paths]

## Examples

Two hand-made traces: they agree until the quality check, whose replies differ, and the
next step cites it. That passes; the same pair with the wrong exits does not.

In [ ]:
from agentic_forecast_review.schemas import (
    CheckQualityArgs,
    Cost,
    DayRange,
    ResolveContextArgs,
    ToolCall,
)

days = DayRange(start_day=169, end_day=196)


def trace(missing: int, last: str) -> tuple[TraceStep, ...]:
    def reply(call_id: str, fields: dict[str, Any]) -> ToolResult:
        return ToolResult(
            call_id=call_id,
            ok=True,
            fields=fields,
            result_hash=result_hash(True, None, fields, (), ()),
        )

    context = {"snapshot_id": f"snap_{missing}", "n_series": 30}
    quality = {"ca_2_foods__missing_days": missing}
    return (
        TraceStep(
            turn=1,
            summary="context",
            call=ToolCall(call_id="ctx1", args=ResolveContextArgs()),
            result=reply("ctx1", context),
        ),
        TraceStep(
            turn=2,
            summary="quality",
            observation_refs=("ctx1",),
            call=ToolCall(
                call_id="dq2", args=CheckQualityArgs(series=("CA_2_FOODS",), days=days)
            ),
            result=reply("dq2", quality),
        ),
        TraceStep(turn=3, summary=last, observation_refs=("dq2",)),
    )


def packet(state: TerminalState, steps: tuple[TraceStep, ...]) -> Packet:
    calls = sum(step.call is not None for step in steps)
    return Packet(
        case_id="x",
        runner="agent",
        snapshot_id="x",
        terminal_state=state,
        trace=steps,
        cost=Cost(turns=len(steps), calls=calls, usd=0, seconds=0),
    )


stop = packet(TerminalState.STOP_UNTRUSTWORTHY, trace(4, "stop"))
answer_steps = trace(0, "answer")
answer = packet(
    TerminalState.STOP_UNTRUSTWORTHY,
    answer_steps[:2]
    + (
        TraceStep(
            turn=3,
            summary="look closer",
            observation_refs=("dq2",),
            call=ToolCall(call_id="ctx3", args=ResolveContextArgs()),
            result=answer_steps[0].result.model_copy(update={"call_id": "ctx3"}),
        ),
        TraceStep(turn=4, summary="answer"),
    ),
)
expected = (TerminalState.STOP_UNTRUSTWORTHY, TerminalState.ANSWER)
check = check_pair(
    stop, answer.model_copy(update={"terminal_state": TerminalState.ANSWER}), expected
)
print(check, check.passed)
assert (
    check.passed
    and check.first_different_step == 3
    and check.first_different_observation == 2
)
assert not check_pair(stop, answer, expected).passed

## The live runs

Three runs of each case per attempt, saved under `runs/kill_gate/<attempt>/`. This calls
the model (under a cent per run) and is skipped by CI. At most three attempts were
allowed; each run record carries the hash of the prompt it used.

In [ ]:
#| notest
from agentic_forecast_review.agent import run_agent_async, save_run
from agentic_forecast_review.cases import build_family

family = build_family()
for _ in range(3):
    for case in (family.cases[1], family.cases[5]):
        record = await run_agent_async(case, family.snapshots[case.snapshot_id])
        print(save_run(record, Path("../runs/kill_gate"), split="attempt_3"))

## The result

The runs of each attempt are paired in order and checked by the adopted criterion
(`lag=1`, decision 0003) and, always reported alongside, by the plan's original one
(`lag=0`).

Attempt 3 is replayed offline first (every reply must match its fingerprint). Attempts
1 and 2 were recorded before `compare_runs` reported horizons, so today's tools give
different replies and replay refuses them; they are read from their saved packets.

In [ ]:
from agentic_forecast_review.agent import replay
from agentic_forecast_review.cases import build_family

family = build_family()
case_a, case_b = family.cases[1], family.cases[5]
snap_a, snap_b = (
    family.snapshots[case_a.snapshot_id],
    family.snapshots[case_b.snapshot_id],
)
verdicts = {}
for attempt in ("attempt_1", "attempt_2", "attempt_3"):
    root = Path("../runs/kill_gate") / attempt
    runs = list(
        zip(
            load_runs(root, case_a.case_id),
            load_runs(root, case_b.case_id),
            strict=True,
        )
    )
    if attempt == "attempt_3":
        pairs = [
            (replay(ra, case_a, snap_a), replay(rb, case_b, snap_b)) for ra, rb in runs
        ]
    else:
        pairs = [
            (Packet.model_validate(ra["packet"]), Packet.model_validate(rb["packet"]))
            for ra, rb in runs
        ]
    strict = [check_pair(a, b, expected, lag=0) for a, b in pairs]
    lagged = [check_pair(a, b, expected) for a, b in pairs]
    verdicts[attempt] = (gate(strict), gate(lagged))
    print(
        f"{attempt}: adopted criterion {'PASS' if gate(lagged) else 'FAIL'}, "
        f"original criterion {'pass' if gate(strict) else 'fail'}"
    )
    for number, ((a, b), check) in enumerate(zip(pairs, strict, strict=True), start=1):
        paths = [
            " > ".join(s.call.args.tool if s.call else "finish" for s in p.trace)
            for p in (a, b)
        ]
        print(
            f"  run {number}: split at step {check.first_different_step}, first "
            f"differing observation at step {check.first_different_observation}, "
            f"exits {check.exits}"
        )
        print(f"    case 2: {paths[0]}")
        print(f"    case 6: {paths[1]}")
assert verdicts["attempt_3"] == (False, True)